<a href="https://colab.research.google.com/github/cras-lab/LangChain/blob/main/NewsBriefing_FromRSS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

필요한 라이브러리를 설치한다.

In [ ]:
pip install -U -q langchain langchain-core langchain-openai langchain-anthropic langchain-google-genai feedparser

환경변수들을 설정한다.

In [2]:
LLM_PROVIDER = "openai" # claude 또는 gemini
KEYFROM_USERDATA = True  # 외부에서 입력하려면 False 로
TEMPERATURE = 0.2

필요한 모듈을 임포트 한다.

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import feedparser

API 키를 입력한다.

In [4]:
if KEYFROM_USERDATA:
  from google.colab import userdata
  API_KEY = userdata.get('OPENAI').strip() # 자신의 보관키에 따라 수정
else:
  import getpass
  API_KEY = getpass.getpass('Enter API key: ')

선택에 따라 해당 LLM을 생성한다.

In [5]:
if LLM_PROVIDER == "openai":
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(
        model="gpt-5-nano",
        temperature=TEMPERATURE,
        api_key=API_KEY,
      )

elif LLM_PROVIDER == "claude":
    from langchain_anthropic import ChatAnthropic

    llm = ChatAnthropic(
        model="claude-sonnet-4-6",
        temperature=TEMPERATURE,
        api_key=API_KEY,
      )


elif LLM_PROVIDER == "gemini":
    from langchain_google_genai import ChatGoogleGenerativeAI

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=TEMPERATURE,
        api_key=API_KEY,
      )


분야별로 연합뉴스 URL을 설정한다.

In [6]:
RSS_URLS = {
    "경제": "https://www.yna.co.kr/rss/economy.xml",
    "산업": "https://www.yna.co.kr/rss/industry.xml",
    "마켓": "https://www.yna.co.kr/rss/market.xml",
    "국제": "https://www.yna.co.kr/rss/international.xml",
    "건강": "https://www.yna.co.kr/rss/health.xml",
}

실습에서는 다음의 10개 주제 분야와 각 키워드를 설정한다.

In [7]:
TOPICS = {
    "금융": ["금융", "은행", "증권", "보험", "대출", "금리", "환율", "코스피", "코스닥"],
    "AI": ["AI", "인공지능", "생성형", "LLM", "챗봇", "데이터센터", "클라우드", "GPU"],
    "반도체": ["반도체", "HBM", "D램", "낸드", "파운드리", "삼성전자", "SK하이닉스"],
    "조선": ["조선", "선박", "LNG선", "해운", "수주", "HD현대", "한화오션", "삼성중공업"],
    "바이오": ["바이오", "제약", "신약", "임상", "FDA", "의약품", "셀트리온", "삼성바이오"],
    "자동차": ["자동차", "현대차", "기아", "전기차", "배터리", "이차전지"],
    "에너지": ["에너지", "원전", "전력", "전기요금", "LNG", "석유", "유가", "수소"],
    "부동산": ["부동산", "아파트", "주택", "전세", "월세", "분양", "건설", "PF"],
    "소비": ["소비", "물가", "유통", "마트", "백화점", "편의점", "식품"],
    "무역": ["수출", "수입", "무역", "관세", "미국", "중국", "EU", "공급망"],
}

각 분야별로 URL에 접속하여 자료를 수집한 후 리스트로 만든다.

In [8]:
headlines = []

for category, url in RSS_URLS.items():
    rss = feedparser.parse(url)

    for entry in rss.entries[:30]:
        title = entry.title

        headlines.append(title)

수집된 헤드라인들을 출력해 보자.

In [ ]:
display( headlines )

 수집된 헤드라인들을 10개 분야로 각 5개 이하로 분류한다.

In [12]:
topic_headlines = {}

for topic, keywords in TOPICS.items():
    topic_headlines[topic] = []

    for title in headlines:
        if any(keyword.lower() in title.lower() for keyword in keywords):
            topic_headlines[topic].append(title)

        if len(topic_headlines[topic]) >= 5:
            break

분류된 것을 출력해 보자.

In [ ]:
display( topic_headlines )

LLM에 넣을 텍스트를 생성한다.

In [15]:
# 3. LLM에 넣을 텍스트 생성
news_text = ""

for topic, titles in topic_headlines.items():
    news_text += f"\n[{topic}]\n"

    if len(titles) == 0:
        news_text += "- 관련 헤드라인 없음\n"
    else:
        for title in titles:
            news_text += f"- {title}\n"

구성된 LLM 프롬프트 내용을 보자

In [ ]:
print(news_text)

요약을 위한 LLM 프롬프트를 구성한다.

In [17]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 경제 뉴스 요약 담당자다. RSS 헤드라인만 보고 분야별 중요 내용을 요약한다."),
    ("human", """
아래는 분야별 뉴스 헤드라인이다.

{news_text}

각 분야별로 다음 형식으로 요약해라.

[분야명]
- 핵심 이슈:
- 중요 헤드라인:
- 관찰 포인트:

헤드라인에 없는 내용은 추측하지 마라.
간결하게 작성해라.
""")
])

LangChain 호출을 위한 파이프라인을 구성한다.

In [18]:
parser = StrOutputParser()

chain = prompt | llm | parser

LLM을 호출한다.

In [19]:
result = chain.invoke({
    "news_text": news_text
})

결과를 출력해 보자.

In [ ]:
print(result)